# Análise de Componentes Principais (PCA)
## Notebook 2 · Um índice de desenvolvimento construído pelos dados

**Data Science · Prof. Eduardo Pécora · UFPR**

---

No notebook anterior vimos, com as notas de uma turma, o que a PCA faz. Agora a aplicamos a um problema real: **resumir 12 indicadores sociais dos 27 estados brasileiros em poucos números**.

> **Revisão rápida**
> - A PCA **gira os eixos**: a 1ª componente (PC1) aponta para onde os dados mais variam; a PC2 é perpendicular e fica com o que sobrou.
> - Cada componente é uma combinação das variáveis; os pesos se chamam **cargas**.
> - A variância total não muda: ela só se **concentra** nas primeiras componentes.

| Parte | Pergunta | Tempo sugerido |
|---|---|---|
| **1** | Por que é preciso padronizar antes da PCA? | ~5 min |
| **2** | Dá para criar um índice de desenvolvimento a partir dos dados? | ~25 min |

Sequência da aula:
1. `PPGOLD_DS_9_1_PCA_Intuicao.ipynb` — intuição (notas da turma)
2. **`PPGOLD_DS_9_2_PCA_Atlas.ipynb`** — este notebook (indicadores dos estados)
3. `PPGOLD_DS_9_3_PCA_Modelos.ipynb` — PCA dentro de um modelo preditivo

### Como trabalhar
- As células com **Tarefa N** são suas: leia o enunciado e escreva o código na célula logo abaixo.
- As demais células já estão prontas: apenas execute.
- Onde aparece **Sua resposta**, escreva em uma ou duas frases o que você concluiu.

#### Começamos importando as bibliotecas e fixando a semente aleatória

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(31415)
sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)

---
# Parte 1 — Por que padronizar?

Agora vamos aos dados reais: **indicadores sociais das 27 Unidades da Federação** no Censo 2010, do *Atlas do Desenvolvimento Humano no Brasil* (PNUD, Ipea e Fundação João Pinheiro).

| Coluna | Significado | Unidade |
|---|---|---|
| `esperanca_vida` | Esperança de vida ao nascer | anos |
| `mortalidade_infantil` | Óbitos antes de 1 ano por mil nascidos vivos | por mil |
| `fecundidade` | Filhos por mulher | filhos |
| `anos_estudo_esperados` | Anos de estudo esperados aos 18 anos | anos |
| `analfabetismo_15mais` | % de analfabetos com 15 anos ou mais | % |
| `superior_25mais` | % com superior completo, 25 anos ou mais | % |
| `renda_per_capita` | Renda domiciliar per capita média | R$ de ago/2010 |
| `gini` | Índice de Gini (desigualdade de renda) | 0 a 1 |
| `pct_pobres` | % de pobres (renda per capita ≤ R$ 140) | % |
| `desemprego` | Taxa de desocupação, 18 anos ou mais | % |
| `trabalho_formal` | % dos ocupados com trabalho formal | % |
| `agua_esgoto_inadequado` | % em domicílios com água e esgoto inadequados | % |
| `idhm` | IDH Municipal oficial do estado (**não entra na PCA**) | 0 a 1 |

> **O IDHM fica de fora de propósito.** No fim da Parte 2 vamos comparar o índice que a PCA constrói sozinha com o IDHM oficial.

#### Lendo os dados

Coloque o arquivo `atlas_brasil_uf_2010.csv` na mesma pasta do notebook (no Colab, use o ícone de pasta à esquerda → *upload*), ou troque o caminho abaixo pela URL do repositório da disciplina.

In [13]:
ARQUIVO = "https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Dados/atlas_brasil_uf_2010.csv"   # ou a URL raw do GitHub da disciplina

df = pd.read_csv(ARQUIVO)
indicadores = df.columns[3:-1]          # as 12 variáveis (sem sigla, estado, região e IDHM)

print(df.shape)
print(list(indicadores))
df.head()

(27, 16)
['esperanca_vida', 'mortalidade_infantil', 'fecundidade', 'anos_estudo_esperados', 'analfabetismo_15mais', 'superior_25mais', 'renda_per_capita', 'gini', 'pct_pobres', 'desemprego', 'trabalho_formal', 'agua_esgoto_inadequado']


,sigla,estado,regiao,esperanca_vida,mortalidade_infantil,fecundidade,anos_estudo_esperados,analfabetismo_15mais,superior_25mais,renda_per_capita,gini,pct_pobres,desemprego,trabalho_formal,agua_esgoto_inadequado,idhm
0,RO,Rondônia,Norte,72.97,18.02,2.16,9.20,8.73,8.04,670.82,0.56,14.80,5.19,51.72,12.43,0.690
1,AC,Acre,Norte,71.63,23.01,2.95,8.69,16.48,8.98,522.15,0.63,29.46,7.46,46.45,28.09,0.663
2,AM,Amazonas,Norte,73.30,17.01,2.59,8.54,9.84,8.23,539.80,0.65,30.78,9.55,44.56,20.98,0.674
3,RR,Roraima,Norte,73.51,16.11,2.41,8.69,10.33,10.16,605.59,0.63,26.65,7.60,47.99,12.44,0.707
4,PA,Pará,Norte,72.36,20.29,2.50,8.49,11.74,6.21,446.76,0.62,32.33,9.03,37.38,29.05,0.646


### Tarefa 1 — As escalas são comparáveis?
1. Use `.describe()` nos 12 indicadores.
2. Calcule a **variância** de cada indicador e ordene da maior para a menor (`.var().sort_values(ascending=False)`).

Compare a variância de `renda_per_capita` com a de `gini`. Lembrando do notebook das notas: o que a PCA vai "achar" que é a direção mais importante?

In [14]:
# Tarefa 1 — seu código aqui

**Sua resposta:** _escreva aqui_

### Tarefa 2 — PCA sem e com padronização
1. Ajuste `PCA()` aos 12 indicadores **sem padronizar** e mostre:
   - `explained_variance_ratio_[:3]`
   - as cargas da PC1 como uma `pd.Series` com os nomes dos indicadores.
2. Repita **padronizando antes** com `StandardScaler().fit_transform(...)`.

O que muda nas cargas da PC1?

In [15]:
# Tarefa 2 — seu código aqui

**Sua resposta:** _escreva aqui_

---
# Parte 2 — Um índice de desenvolvimento construído pelos dados

Daqui em diante usamos os dados **padronizados**.

In [16]:
X_pad = StandardScaler().fit_transform(df[indicadores])
pca = PCA().fit(X_pad)

nomes_pc = [f"PC{i+1}" for i in range(len(indicadores))]
print(f"{X_pad.shape[1]} variáveis  ->  {pca.n_components_} componentes possíveis")

12 variáveis  ->  12 componentes possíveis


### Tarefa 3 — Quanta informação cada componente guarda?
1. Monte um `DataFrame` chamado `tabela` com uma linha por componente e as colunas:
   - `autovalor` → `pca.explained_variance_`
   - `% variância` → `pca.explained_variance_ratio_ * 100`
   - `% acumulada` → a soma acumulada da coluna anterior (`.cumsum()`)
2. Faça um gráfico com **barras** para a `% variância` e uma **linha** para a `% acumulada` (os dois em %, no mesmo eixo).

In [17]:
# Tarefa 3 — seu código aqui
tabela = ...

### Tarefa 4 — Quantas componentes manter?
Não existe resposta única; usamos critérios e comparamos.

1. **Critério da variância acumulada:** quantas componentes são necessárias para chegar a **80%**? E a **90%**?
2. **Critério de Kaiser:** com dados padronizados, cada variável original tem variância 1. Uma componente só "vale a pena" se guardar **mais que uma variável**, isto é, se tiver **autovalor > 1**. Quantas componentes passam?

Dica: `(tabela["% acumulada"] < 80).sum() + 1` e `(tabela["autovalor"] > 1).sum()`.

In [18]:
# Tarefa 4 — seu código aqui

**Sua resposta:** _escreva aqui_

### Tarefa 5 — O que cada componente significa?
1. Monte um `DataFrame` chamado `cargas` com as cargas das **3 primeiras** componentes: linhas = indicadores, colunas = `PC1`, `PC2`, `PC3`.
   Dica: `pca.components_[:3].T`.
2. Execute a célula seguinte (já pronta), que desenha o mapa de calor das cargas.
3. Responda: quais indicadores têm carga **positiva** forte na PC1? Quais têm carga **negativa**? Que nome você daria à PC1? E à PC2?

In [19]:
# Tarefa 5 — seu código aqui
cargas = ...

In [20]:
if not isinstance(globals().get("cargas"), pd.DataFrame):
    print("Complete a Tarefa 5: crie o DataFrame cargas antes de rodar esta célula.")
else:
    plt.figure(figsize=(6, 7))
    sns.heatmap(cargas, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
                cbar_kws={"label": "carga"})
    plt.title("Cargas: o peso de cada indicador em cada componente")
    plt.show()

Complete a Tarefa 5: crie o DataFrame cargas antes de rodar esta célula.


**Sua resposta:** _escreva aqui_

### Tarefa 6 — Onde fica cada estado?
1. Crie um `DataFrame` chamado `scores` com as duas primeiras colunas de `pca.transform(X_pad)` (nomes `PC1` e `PC2`) e acrescente as colunas `sigla` e `regiao` de `df`.
2. Execute a célula seguinte, que desenha o **biplot**: os estados (pontos) e as cargas (setas) no mesmo gráfico.
3. Responda: quais estados ficam nos extremos da PC1? Por que o **Distrito Federal** fica isolado no alto (ou embaixo) da PC2?

In [21]:
# Tarefa 6 — seu código aqui
scores = ...

In [22]:
if not isinstance(globals().get("scores"), pd.DataFrame):
    raise SystemExit("Complete a Tarefa 6: crie o DataFrame scores antes de rodar esta célula.")

cores = {"Norte": "#009E73", "Nordeste": "#D55E00", "Centro-Oeste": "#E69F00",
         "Sudeste": "#0072B2", "Sul": "#CC79A7"}
marcadores = {"Norte": "o", "Nordeste": "s", "Centro-Oeste": "D", "Sudeste": "^", "Sul": "v"}

fig, ax = plt.subplots(figsize=(12, 9))
for reg, grupo in scores.groupby("regiao"):
    ax.scatter(grupo["PC1"], grupo["PC2"], s=90, color=cores[reg], marker=marcadores[reg],
               label=reg, edgecolor="white", linewidth=1)
for _, linha in scores.iterrows():
    ax.annotate(linha["sigla"], (linha["PC1"], linha["PC2"]), xytext=(5, 4),
                textcoords="offset points", fontsize=10)

escala = 7   # só para as setas ficarem visíveis na mesma escala dos pontos
for var, (c1, c2) in cargas[["PC1", "PC2"]].iterrows():
    ax.arrow(0, 0, escala * c1, escala * c2, color="grey", alpha=0.8, width=0.015, head_width=0.12)
    ax.text(escala * c1 * 1.1, escala * c2 * 1.1, var, color="dimgrey", fontsize=9, ha="center",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))

ax.axhline(0, color="black", lw=0.5); ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} da variância)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} da variância)")
ax.set_title("Biplot: estados e indicadores no plano das duas primeiras componentes")
ax.legend(title="Região")
plt.show()

SystemExit: Complete a Tarefa 6: crie o DataFrame scores antes de rodar esta célula.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


**Sua resposta:** _escreva aqui_

### Tarefa 7 — Nosso índice × IDHM oficial
1. Crie a coluna `indice_pca` em `df` a partir da PC1, **orientada para que valor maior = mais desenvolvido**.
   Dica: se a carga de `esperanca_vida` na PC1 for negativa, multiplique a PC1 por −1. O código `np.sign(cargas.loc["esperanca_vida", "PC1"])` devolve esse sinal.
2. Mostre o **ranking** dos estados pelo `indice_pca` (`sort_values`).
3. Calcule a **correlação** entre `indice_pca` e `idhm` e faça um gráfico de dispersão das duas, com as siglas.

In [ ]:
# Tarefa 7 — seu código aqui

**Sua resposta:** _escreva aqui_

---
# Para levar

| Faça | Evite |
|---|---|
| Padronize variáveis com unidades diferentes | PCA em variáveis categóricas codificadas como números |
| Escolha *k* com mais de um critério (acumulada, Kaiser, cotovelo) | Escolher *k* "no olho" sem justificar |
| Interprete as cargas antes de usar os scores | Tratar as PCs como "caixa-preta" |
| Lembre que o sinal é arbitrário | Concluir algo pelo sinal sozinho |
| Use PCA para resumir, visualizar e reduzir colinearidade | Esperar que a PCA capture relações não lineares (ver t-SNE, UMAP) |

**Próximo passo:** no notebook `PPGOLD_DS_9_3_PCA_Modelos.ipynb` usamos a PCA **dentro de um modelo preditivo**, sem cometer vazamento de dados.

## Bibliografia

[1] scikit-learn: [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

[2] PNUD, Ipea, FJP. *Atlas do Desenvolvimento Humano no Brasil* — [atlasbrasil.org.br](http://www.atlasbrasil.org.br)

[3] JAMES, G. et al. *An Introduction to Statistical Learning*, cap. 12 (Unsupervised Learning).

## Autor

<a href="https://www.linkedin.com/in/eduardopecora/" target="_blank">Eduardo Pecora</a>

## Log de modificações

| Data | Versão | Modificado por | Descrição |
| ----------------- | ------- | ---------- | ---------------------------------- |
| 24-10-2021 | 1.0 | Eduardo Pecora | Criação (aula única de PCA) |
| 09-11-2026 | 2.0 | Eduardo Pecora | Notebook próprio: padronização e interpretação com os dados do Atlas Brasil |